# 01 — BCRD Exchange Rates | Gold KPIs

**Project:** RD Economic & Financial Intelligence Platform

## Objective
Calculate the validated foreign-exchange KPIs **FX-001 to FX-009** from Silver, following the approved calculation decisions **D1–D4**, in a model ready for Databricks SQL and Power BI.

## Lineage
| Layer | Object | Grain |
|---|---|---|
| Source | `silver.bcrd_exchange_rates_daily` | Business day |
| Target (daily) | `gold.exchange_rate_metrics` | Date × Currency pair × Rate type |
| Target (monthly) | `gold.exchange_rate_monthly_metrics` | Month × Currency pair × Rate type |

## KPI mapping
| KPI | Column | Table |
|---|---|---|
| FX-001 / FX-002 | `rate` (`rate_type` = buy / sell) | daily |
| FX-003 Daily FX Change | `daily_change_pct` | daily |
| FX-004 MoM FX Change | `mom_change_pct` | monthly |
| FX-005 YoY FX Change | `yoy_change_pct` | monthly |
| FX-006 YTD FX Change | `ytd_change_pct` | daily and monthly |
| FX-007 30-Day Moving Average | `ma30_rate` | daily |
| FX-008 52-Week High | `high_52w_rate` | daily |
| FX-009 52-Week Low | `low_52w_rate` | daily |

## Calculation decisions
- **D1** Conventional change `(Rate_t / Rate_ref − 1) × 100`. Positive = rate up = **peso depreciation**. BCRD official values (reciprocal method, opposite sign) stay as `bcrd_*_ref_pct` and are used only as a reconciliation check.
- **D2** Month reference = rate on the last available business day of the month.
- **D3** Moving average = trailing 30 business-day observations, current day included.
- **D4** Windows without enough history return NULL with an `is_full_*` flag set to false.

**Status:** Development — not production-ready.

In [0]:
# ============================================================
# Step 1 — Configuration
# ============================================================
from datetime import datetime, timezone
import uuid

from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "rd_economic_intelligence"
SOURCE_TABLE = f"{CATALOG}.silver.bcrd_exchange_rates_daily"
DAILY_TABLE = f"{CATALOG}.gold.exchange_rate_metrics"
MONTHLY_TABLE = f"{CATALOG}.gold.exchange_rate_monthly_metrics"
RULE_RESULTS_TABLE = f"{CATALOG}.quality.dq_rule_results"

MA_WINDOW_OBS = 30          # D3
WINDOW_52W_DAYS = 364       # 52 weeks, current day included
BCRD_REF_TOLERANCE = 0.15   # BCRD publishes 1 decimal; tolerance covers rounding

RUN_ID = str(uuid.uuid4())
RUN_TS = datetime.now(timezone.utc)
print(f"Run ID: {RUN_ID}")

In [0]:
# ============================================================
# Step 2 — Read Silver and reshape to long format (one row per rate type)
# ============================================================
silver = spark.table(SOURCE_TABLE)

long_df = (
    silver.select(
        "rate_date", "currency_pair", "year", "month",
        F.explode(F.array(
            F.struct(F.lit("buy").alias("rate_type"), F.col("buy_rate").alias("rate"),
                     F.col("bcrd_buy_ytd_apre_depre_ref_pct").alias("bcrd_ytd_ref_pct"),
                     F.col("bcrd_buy_yoy_apre_depre_ref_pct").alias("bcrd_yoy_ref_pct")),
            F.struct(F.lit("sell").alias("rate_type"), F.col("sell_rate").alias("rate"),
                     F.col("bcrd_sell_ytd_apre_depre_ref_pct").alias("bcrd_ytd_ref_pct"),
                     F.col("bcrd_sell_yoy_apre_depre_ref_pct").alias("bcrd_yoy_ref_pct")),
        )).alias("r"),
    )
    .select("rate_date", "currency_pair", "year", "month", "r.*")
    .withColumn("rate", F.col("rate").cast("double"))
)

silver_count = silver.count()
print(f"Silver rows: {silver_count:,} -> long rows: {long_df.count():,}")

In [0]:
# ============================================================
# Step 3 — Daily KPIs (FX-001, FX-002, FX-003, FX-006, FX-007, FX-008, FX-009)
# ============================================================
series = Window.partitionBy("currency_pair", "rate_type").orderBy("rate_date")
ma_window = series.rowsBetween(-(MA_WINDOW_OBS - 1), 0)
day_number = F.datediff("rate_date", F.lit("1970-01-01"))
w52 = (Window.partitionBy("currency_pair", "rate_type")
       .orderBy(day_number).rangeBetween(-(WINDOW_52W_DAYS - 1), 0))
whole_series = Window.partitionBy("currency_pair", "rate_type")

# Last available rate of each year -> reference for next year's YTD (FX-006)
year_end = (
    long_df.withColumn("_rn", F.row_number().over(
        Window.partitionBy("currency_pair", "rate_type", "year").orderBy(F.col("rate_date").desc())))
    .filter("_rn = 1")
    .select("currency_pair", "rate_type",
            (F.col("year") + 1).alias("year"),
            F.col("rate_date").alias("prev_year_end_date"),
            F.col("rate").alias("prev_year_end_rate"))
)

daily = (
    long_df
    .withColumn("prev_rate", F.lag("rate").over(series))
    .withColumn("daily_change_pct", (F.col("rate") / F.col("prev_rate") - 1) * 100)
    .withColumn("ma30_rate", F.avg("rate").over(ma_window))
    .withColumn("_ma_obs", F.count("rate").over(ma_window))
    .withColumn("high_52w_rate", F.max("rate").over(w52))
    .withColumn("low_52w_rate", F.min("rate").over(w52))
    .withColumn("_series_start", F.min("rate_date").over(whole_series))
    .join(year_end, ["currency_pair", "rate_type", "year"], "left")
    .withColumn("ytd_change_pct", (F.col("rate") / F.col("prev_year_end_rate") - 1) * 100)
    # D4 completeness flags
    .withColumn("is_full_window_ma30", F.col("_ma_obs") == MA_WINDOW_OBS)
    .withColumn("is_full_window_52w",
                F.col("_series_start") <= F.date_sub("rate_date", WINDOW_52W_DAYS - 1))
    .withColumn("is_full_window_ytd", F.col("prev_year_end_rate").isNotNull())
    # D4: NULL when the window is incomplete
    .withColumn("ma30_rate", F.when(F.col("is_full_window_ma30"), F.col("ma30_rate")))
    .withColumn("high_52w_rate", F.when(F.col("is_full_window_52w"), F.col("high_52w_rate")))
    .withColumn("low_52w_rate", F.when(F.col("is_full_window_52w"), F.col("low_52w_rate")))
    # D1: BCRD official equivalent of our YTD, to reconcile with the published value
    #   BCRD (reciprocal) = (prev_year_end_rate / rate - 1) * 100
    .withColumn("ytd_bcrd_method_pct", (F.col("prev_year_end_rate") / F.col("rate") - 1) * 100)
)

daily_out = daily.select(
    "rate_date", "currency_pair", "rate_type",
    F.col("rate").cast("decimal(10,4)").alias("rate"),
    F.col("prev_rate").cast("decimal(10,4)").alias("prev_rate"),
    F.round("daily_change_pct", 4).cast("decimal(10,4)").alias("daily_change_pct"),
    F.col("prev_year_end_date"),
    F.col("prev_year_end_rate").cast("decimal(10,4)").alias("prev_year_end_rate"),
    F.round("ytd_change_pct", 4).cast("decimal(10,4)").alias("ytd_change_pct"),
    "is_full_window_ytd",
    F.round("ma30_rate", 4).cast("decimal(10,4)").alias("ma30_rate"),
    "is_full_window_ma30",
    F.col("high_52w_rate").cast("decimal(10,4)").alias("high_52w_rate"),
    F.col("low_52w_rate").cast("decimal(10,4)").alias("low_52w_rate"),
    "is_full_window_52w",
    F.round("ytd_bcrd_method_pct", 4).cast("decimal(10,4)").alias("ytd_bcrd_method_pct"),
    F.col("bcrd_ytd_ref_pct"),
    F.col("bcrd_yoy_ref_pct"),
    "year", "month",
    F.lit(RUN_ID).alias("_gold_run_id"),
    F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"),
).orderBy("rate_type", "rate_date")

In [0]:
# ============================================================
# Step 4 — Monthly KPIs (FX-004, FX-005, month-end FX-006)
# ============================================================
month_rank = Window.partitionBy("currency_pair", "rate_type", "year", "month").orderBy(F.col("rate_date").desc())
monthly_series = Window.partitionBy("currency_pair", "rate_type").orderBy("month_start")

month_stats = (
    daily.groupBy("currency_pair", "rate_type", "year", "month")
    .agg(F.avg("rate").alias("month_avg_rate"), F.count("*").alias("business_days"))
)

monthly = (
    daily.withColumn("_rn", F.row_number().over(month_rank)).filter("_rn = 1")   # D2: month end
    .select("currency_pair", "rate_type", "year", "month",
            F.col("rate_date").alias("month_end_date"),
            F.col("rate").alias("month_end_rate"),
            "ytd_change_pct", "is_full_window_ytd")
    .join(month_stats, ["currency_pair", "rate_type", "year", "month"])
    .withColumn("month_start", F.make_date("year", "month", F.lit(1)))
    .withColumn("_prev_month_start", F.lag("month_start", 1).over(monthly_series))
    .withColumn("_prev_month_rate", F.lag("month_end_rate", 1).over(monthly_series))
    .withColumn("_yago_month_start", F.lag("month_start", 12).over(monthly_series))
    .withColumn("_yago_month_rate", F.lag("month_end_rate", 12).over(monthly_series))
    # Only compare with the exact previous month / same month last year
    .withColumn("is_full_window_mom", F.coalesce(F.months_between("month_start", "_prev_month_start") == 1, F.lit(False)))
    .withColumn("is_full_window_yoy", F.coalesce(F.months_between("month_start", "_yago_month_start") == 12, F.lit(False)))
    # First and last month of the source extract do not cover the whole month
    # (e.g. the file ends on 2026-09-18). Their month_end_rate is the latest available
    # rate, so KPIs for those months are month-to-date and must be labelled as such.
    .withColumn("_first_month", F.min("month_start").over(Window.partitionBy("currency_pair", "rate_type")))
    .withColumn("_last_month", F.max("month_start").over(Window.partitionBy("currency_pair", "rate_type")))
    .withColumn("is_partial_month", (F.col("month_start") == F.col("_first_month")) | (F.col("month_start") == F.col("_last_month")))
    .withColumn("mom_change_pct", F.when(F.col("is_full_window_mom"),
                (F.col("month_end_rate") / F.col("_prev_month_rate") - 1) * 100))
    .withColumn("yoy_change_pct", F.when(F.col("is_full_window_yoy"),
                (F.col("month_end_rate") / F.col("_yago_month_rate") - 1) * 100))
)

monthly_out = monthly.select(
    "month_start", "currency_pair", "rate_type", "month_end_date",
    F.col("month_end_rate").cast("decimal(10,4)").alias("month_end_rate"),
    F.round("month_avg_rate", 4).cast("decimal(10,4)").alias("month_avg_rate"),
    F.col("business_days").cast("int").alias("business_days"),
    "is_partial_month",
    F.round("mom_change_pct", 4).cast("decimal(10,4)").alias("mom_change_pct"),
    "is_full_window_mom",
    F.round("yoy_change_pct", 4).cast("decimal(10,4)").alias("yoy_change_pct"),
    "is_full_window_yoy",
    F.round("ytd_change_pct", 4).cast("decimal(10,4)").alias("ytd_change_pct"),
    "is_full_window_ytd",
    "year", "month",
    F.lit(RUN_ID).alias("_gold_run_id"),
    F.lit(RUN_TS).cast("timestamp").alias("_gold_processed_ts"),
).orderBy("rate_type", "month_start")

In [0]:
# ============================================================
# Step 5 — Write Gold tables (full refresh)
# ============================================================
for df, table in [(daily_out, DAILY_TABLE), (monthly_out, MONTHLY_TABLE)]:
    (df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(table))

daily_count = spark.table(DAILY_TABLE).count()
monthly_count = spark.table(MONTHLY_TABLE).count()
print(f"{DAILY_TABLE}: {daily_count:,} rows")
print(f"{MONTHLY_TABLE}: {monthly_count:,} rows")

In [0]:
# ============================================================
# Step 6 — Documentation in Unity Catalog
# ============================================================
def document(table, table_comment, columns):
    spark.sql(f"COMMENT ON TABLE {table} IS '{table_comment}'")
    for col, comment in columns.items():
        spark.sql(f"ALTER TABLE {table} ALTER COLUMN {col} COMMENT '{comment}'")

document(DAILY_TABLE,
    "Gold: USD/DOP daily FX KPIs (FX-001, 002, 003, 006, 007, 008, 009). Grain: date x currency pair x rate type. Rate up = peso depreciation.",
    {
        "rate": "FX-001/FX-002: BCRD published rate, DOP per 1 USD",
        "daily_change_pct": "FX-003: % change vs previous business day. Positive = peso depreciation",
        "ytd_change_pct": "FX-006: % change vs last business day of previous year. Positive = peso depreciation",
        "ma30_rate": "FX-007: average of the trailing 30 business-day observations; NULL if fewer than 30",
        "high_52w_rate": "FX-008: highest rate in trailing 364 calendar days; NULL until 52 weeks of history",
        "low_52w_rate": "FX-009: lowest rate in trailing 364 calendar days; NULL until 52 weeks of history",
        "ytd_bcrd_method_pct": "Our YTD restated with the BCRD reciprocal method, used to reconcile with bcrd_ytd_ref_pct",
        "bcrd_ytd_ref_pct": "BCRD official accumulated appreciation(+)/depreciation(-), reference only",
        "bcrd_yoy_ref_pct": "BCRD official year-on-year appreciation(+)/depreciation(-), reference only",
    })

document(MONTHLY_TABLE,
    "Gold: USD/DOP monthly FX KPIs (FX-004, FX-005, month-end FX-006). Grain: month x currency pair x rate type. Rate up = peso depreciation.",
    {
        "month_end_rate": "Rate on the last available business day of the month (D2)",
        "month_avg_rate": "Average daily rate of the month, for comparisons with monthly CPI",
        "is_partial_month": "True for the first and last month of the source extract: values are month-to-date, not full month",
        "mom_change_pct": "FX-004: % change of month-end rate vs previous month-end",
        "yoy_change_pct": "FX-005: % change of month-end rate vs same month previous year; NULL until 12 months of history",
        "ytd_change_pct": "FX-006 at month end: % change vs last business day of previous year",
    })

print("Documentation applied")

In [0]:
# ============================================================
# Step 7 — Validation and reconciliation
# ============================================================
d = spark.table(DAILY_TABLE)
m = spark.table(MONTHLY_TABLE)

checks = []

def add(name, rtype, tested, failed):
    status = "PASS" if failed == 0 else ("FAIL" if rtype == "enforced" else "WARN")
    checks.append((RUN_ID, DAILY_TABLE, name, rtype, int(tested), int(failed), status, RUN_TS))

# Completeness: 2 rate types per Silver day
add("row_count_matches_silver_x2", "enforced", silver_count * 2, abs(daily_count - silver_count * 2))
add("unique_date_currency_ratetype", "enforced", daily_count,
    daily_count - d.select("rate_date", "currency_pair", "rate_type").distinct().count())
# Windows: complete windows must have values, incomplete must be NULL (D4)
add("ma30_null_iff_incomplete", "enforced", daily_count,
    d.filter(F.col("ma30_rate").isNull() == F.col("is_full_window_ma30")).count())
add("range52w_null_iff_incomplete", "enforced", daily_count,
    d.filter(F.col("high_52w_rate").isNull() == F.col("is_full_window_52w")).count())
add("low_not_above_high_52w", "enforced", daily_count,
    d.filter(F.col("low_52w_rate") > F.col("high_52w_rate")).count())
add("rate_within_52w_range", "enforced", daily_count,
    d.filter((F.col("rate") < F.col("low_52w_rate")) | (F.col("rate") > F.col("high_52w_rate"))).count())

# D1 reconciliation: our YTD restated with BCRD method vs BCRD published value
comparable = d.filter(F.col("is_full_window_ytd") & F.col("bcrd_ytd_ref_pct").isNotNull())
mismatch = comparable.filter(F.abs(F.col("ytd_bcrd_method_pct") - F.col("bcrd_ytd_ref_pct")) > BCRD_REF_TOLERANCE)
add("ytd_reconciles_with_bcrd_official", "observational", comparable.count(), mismatch.count())

spark.createDataFrame(
    checks,
    "run_id string, target_table string, rule_name string, rule_type string, "
    "records_tested bigint, records_failed bigint, status string, run_ts timestamp",
).write.mode("append").saveAsTable(RULE_RESULTS_TABLE)

print("Validation results:")
for c in checks:
    print(f"  {c[6]:<4} {c[2]:<36} ({c[3]}) tested={c[4]:,} failed={c[5]:,}")

if mismatch.count() > 0:
    display(mismatch.select("rate_date", "rate_type", "rate", "prev_year_end_rate",
                            "ytd_bcrd_method_pct", "bcrd_ytd_ref_pct").orderBy("rate_date").limit(20))

failed_enforced = [c[2] for c in checks if c[6] == "FAIL"]
assert not failed_enforced, f"Enforced checks failed: {failed_enforced}"

In [0]:
# ============================================================
# Step 8 — Business snapshot (latest values)
# ============================================================
latest = d.filter(F.col("rate_date") == d.agg(F.max("rate_date")).first()[0])
display(latest.select("rate_date", "rate_type", "rate", "daily_change_pct", "ytd_change_pct",
                      "ma30_rate", "high_52w_rate", "low_52w_rate", "bcrd_ytd_ref_pct"))

latest_month = m.filter(F.col("month_start") == m.agg(F.max("month_start")).first()[0])
display(latest_month.select("month_start", "rate_type", "month_end_date", "is_partial_month",
                            "month_end_rate", "mom_change_pct", "yoy_change_pct"))

In [0]:
import json

def as_dict(rows):
    return [{k: (str(v) if v is not None else None) for k, v in r.asDict().items()} for r in rows]

dbutils.notebook.exit(json.dumps({
    "status": "SUCCESS",
    "run_id": RUN_ID,
    "daily_rows": daily_count,
    "monthly_rows": monthly_count,
    "checks": {c[2]: f"{c[6]} ({c[5]}/{c[4]} failed)" for c in checks},
    "latest_daily": as_dict(latest.select("rate_date", "rate_type", "rate", "daily_change_pct", "ytd_change_pct",
                                          "ma30_rate", "high_52w_rate", "low_52w_rate", "bcrd_ytd_ref_pct",
                                          "ytd_bcrd_method_pct").collect()),
    "partial_months": [str(r[0]) for r in m.filter("is_partial_month").select("month_start").distinct().orderBy("month_start").collect()],
    "latest_month": as_dict(latest_month.select("month_start", "rate_type", "month_end_date", "is_partial_month", "month_end_rate",
                                                "mom_change_pct", "yoy_change_pct").collect()),
    "first_full_52w": str(d.filter("is_full_window_52w").agg(F.min("rate_date")).first()[0]),
    "first_full_yoy": str(m.filter("is_full_window_yoy").agg(F.min("month_start")).first()[0]),
}))